In [ ]:
%pip install -qqq pandas numpy matplotlib ucimlrepo seaborn lightgbm scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, mean_squared_error, root_mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from ucimlrepo import fetch_ucirepo 

In [ ]:
seed = 42

## 1. データセットの読み込み

In [ ]:
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 

In [ ]:
# datasetの特徴量＋予測対象を一括で読み込む
df_raw = wine_quality.data.original

## 2. 前処理

In [ ]:
y_col_names = ["quality", "color"]
x_col_names = df_raw.drop(y_col_names, axis=1).columns

In [ ]:
# 特徴量カラムの確認
x_col_names

In [ ]:
# 前処理用にdf作成
df_source = df_raw.copy()

In [ ]:
# カラム指定で標準化を行う設定
ct = ColumnTransformer(
    transformers=[("num", StandardScaler(), x_col_names)],
    remainder="passthrough",
    verbose_feature_names_out=False,
)

In [ ]:
# 標準化を実行（戻り値は NumPy 配列）
scaled_array = ct.fit_transform(df_source)

In [ ]:
columns = ct.get_feature_names_out()
df_scaled = pd.DataFrame(scaled_array, columns=columns)

In [ ]:
# 標準化されたか確認
df_scaled.head(5)

In [ ]:
df_scaled.dtypes

In [ ]:
df_scaled[x_col_names] = df_scaled[x_col_names].astype("float")
df_scaled[y_col_names[0]] = df_scaled[y_col_names[0]].astype("float")

# remainder="passthrough" の出力はscikit-learn の仕様上、何も処理しない列（passthrough）は、
# 安全性を優先するために object 型の配列として抽出されることがあるので、objectからfloatに戻す

In [ ]:
df_scaled.dtypes

In [ ]:
# カテゴリ変数のcolorを確認
df_scaled["color"].value_counts()

In [ ]:
# colorのラベルエンコーディング
le = LabelEncoder()

df_scaled["color"] = le.fit_transform(df_scaled["color"])

In [ ]:
# ラベルエンコーディングされてるか確認
df_scaled["color"].value_counts()

In [ ]:
# trainとtestに分割する関数の定義

def make_dataset(x_cols, y_col, df, seed):
    
    X_train, X_test, y_train, y_test = train_test_split(
                                            df[x_cols], 
                                            df[y_col], 
                                            test_size=0.2, 
                                            random_state=seed
                                        )

    return (
        X_train.reset_index(drop=True), 
        X_test.reset_index(drop=True), 
        y_train.reset_index(drop=True), 
        y_test.reset_index(drop=True)
    )

In [ ]:
# make_datasetの動作確認
# y_col_names = ["quality", "color"]
y_col = y_col_names[0]
# y_col = y_col_names[1]

make_dataset(x_cols=x_col_names, y_col=y_col, df=df_scaled, seed=seed)


## 3. モデルの学習&評価

### 3.1 回帰

In [ ]:
# lightgbmのインスタンス作成
model_reg_lgb = lgb.LGBMRegressor(
    n_estimators=100, 
    learning_rate=0.1, 
    importance_type="gain",
    random_state=seed
)

In [ ]:
# svrのインスタンス作成
model_reg_svr = SVR()

In [ ]:
# qualityを回帰として予測する

# y_col_names = ["quality", "color"]
y_col = ["quality"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = make_dataset(x_cols=x_col_names,
                                                                y_col=y_col,
                                                                df=df_scaled,
                                                                seed=seed
                                                               )

In [ ]:
# 学習
model_reg_lgb.fit(X_train_reg, y_train_reg)
model_reg_svr.fit(X_train_reg, y_train_reg)

In [ ]:
# lightgbm
y_pred_lgb = model_reg_lgb.predict(X_test_reg)
rmse_lgb = root_mean_squared_error(y_test_reg, y_pred_lgb)

In [ ]:
# svr
y_pred_svr = model_reg_svr.predict(X_test_reg)
rmse_svr = root_mean_squared_error(y_test_reg, y_pred_svr)

In [ ]:
print(f"RMSE:\nlightgbm{rmse_lgb}\nsvr{rmse_svr}")

In [ ]:
# lightgbmでfeature importanceの確認

# 特徴量重要度の DataFrame を作成
df_importance = pd.DataFrame({
    "Feature": X_train_reg.columns,
    "Gain": model_reg_lgb.feature_importances_
})

# 重要度が高い順に並び替え
df_importance = df_importance.sort_values(by="Gain",
                                          ascending=False).reset_index(drop=True)

print(df_importance)

In [ ]:
# 描画サイズの設定 (特徴量の数に合わせて高さを調整してください)
plt.figure(figsize=(10, 6))

# 横棒グラフの描画
sns.barplot(
    x="Gain",
    y="Feature",
    data=df_importance,
    palette="viridis" # 綺麗なグラデーションカラー
)

# グラフの装飾
plt.title("Feature Importance (Gain) of Regression", fontsize=14)
plt.xlabel("Importance (Gain)", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()

# 表示
plt.show()

## 3.2 分類

In [ ]:
model_clf_lgb = lgb.LGBMClassifier(
    boosting_type="gbdt",
    n_estimators=100,
    learning_rate=0.1,
    is_unbalance=True, #不均衡データなので
    importance_type="gain",
    random_state=seed
)

In [ ]:
# svcのインスタンス作成
model_clf_svc = SVC(class_weight="balanced")

In [ ]:
# qualityを回帰として予測する

# y_col_names = ["quality", "color"]
y_col = ["color"]

X_train_clf, X_test_clf, y_train_clf, y_test_clf = make_dataset(x_cols=x_col_names,
                                                                y_col=y_col,
                                                                df=df_scaled,
                                                                seed=seed
                                                               )

In [ ]:
model_clf_lgb.fit(X_train_clf, y_train_clf)
model_clf_svc.fit(X_train_clf, y_train_clf)

In [ ]:
# lightgbm
y_pred_clf_lgb = model_clf_lgb.predict(X_test_clf)
acc_lgb = accuracy_score(y_test_clf, y_pred_clf_lgb)
f1_lgb = f1_score(y_test_clf, y_pred_clf_lgb)
print(acc_lgb, f1_lgb)

In [ ]:
# svc
y_pred_clf_svr = model_clf_svc.predict(X_test_clf)
acc_svr = accuracy_score(y_test_clf, y_pred_clf_svr)
f1_svr = f1_score(y_test_clf, y_pred_clf_svr)
print(acc_svr, f1_svr)

In [ ]:
# cofusion matrix
# lgb
cm_lgb = confusion_matrix(y_test_clf, y_pred_clf_lgb)
cm_lgb

In [ ]:
# cofusion matrix
# svr
cm_svr = confusion_matrix(y_test_clf, y_pred_clf_svr)
cm_svr

In [ ]:
# fature importance

# lightgbmでfeature importanceの確認

# 特徴量重要度の DataFrame を作成
df_importance_clf = pd.DataFrame({
    "Feature": X_train_clf.columns,
    "Gain": model_clf_lgb.feature_importances_
})

# 重要度が高い順に並び替え
df_importance_clf = df_importance_clf.sort_values(by="Gain",
                                          ascending=False).reset_index(drop=True)

print(df_importance_clf)

In [ ]:
# 描画サイズの設定 (特徴量の数に合わせて高さを調整してください)
plt.figure(figsize=(10, 6))

# 横棒グラフの描画
sns.barplot(
    x="Gain",
    y="Feature",
    data=df_importance_clf,
    palette="viridis" # 綺麗なグラデーションカラー
)

# グラフの装飾
plt.title("Feature Importance (Gain) of Classification", fontsize=14)
plt.xlabel("Importance (Gain)", fontsize=12)
plt.ylabel("Features", fontsize=12)
plt.tight_layout()

# 表示
plt.show()